# 11.13 四個字對三個，整串編號就正確嗎？


材料寫 `1010`，預測寫 `1000`。第三位 1 被讀成 0，四位對三位，整串卻不相同。查驗編號要用完整字串，所以要把逐位正確與整串成功分開。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
truth = "1010"
predicted = "1000"
assert len(truth) == len(predicted)  # 本實驗只比較等長且對齊的字串
matched = sum(a == b for a, b in zip(truth, predicted))
print("等長逐位正確率", matched / len(truth))
print("整串正確", truth == predicted)
print("本例一次替換的CER", 1 / len(truth))
print("獨立錯誤假設下五位全對", round(0.9**5, 4))

本例等長且對齊，matched＝3，印逐位 0.75、整串 False；一次替換的字元錯誤率 CER 是 1/4＝0.25。CER 一般用最少插入、刪除、替換次數除以參考長度。程式只算這個已知一次替換例，沒有用 zip 假裝通用編輯距離。

若預測 `10100`，刪多的一個 0，仍是 1/4；若預測 `010`，在開頭補一個 1，也只是一次編輯，不能把錯位的逐位置差異全部算成錯字。參考長度仍 4，不是預測長度。

最後一行假設每位正確率 0.9 且互相獨立，五位全對約 0.5905。這是乘法直覺，真實裁切或字序錯誤可能一起發生，仍要直接量完整字串，不能由逐字平均推算。

完整有序答案最後加 EOS，通知生成停止；它不是圖片上的字。既有 OCR 最後題 30/30 都以 EOS 結束，完整內容卻只有 2/30，通用編輯距離合計 40 次、參考 57 字，CER 約 70.2%。會停止和讀對，仍是兩個能力。

練習改預測為 `1010`，逐位應 1、整串 True；本例 CER 那行也改成零次編輯。再寫 `101`，說明補末尾 0 仍是 1/4，而程式的等長 assert 會拒絕直接拿來算。

<details>
<summary>回顧與查證</summary>

可回顧：[11.12字形與文字標籤](https://birdhackor.github.io/tiny-perceptron-vlm/11.12.html)、[W.2清單與迴圈](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/ocr.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
